In [1]:
#TODO: CHANGE TO ONLY CLEANED WEATHER, READD SOD FOR THAT

In [2]:
# weather (specific time) + accidents
#daily weather --> BR = mist
# https://forecast.weather.gov/glossary.php?

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [4]:
spark = SparkSession.builder.appName("NYC_Twilight_Visibility").getOrCreate()

base_path = "/home/jovyan/work"

In [5]:
weather_df = spark.read.parquet(f"{base_path}/CLEANED_weather_partitioned/")
collision_df = spark.read.parquet(f"{base_path}/CLEANED_vehicle_collisions_partitioned/")

weather_df_filtered = weather_df.filter((weather_df.year == 2023) | (weather_df.year == 2024) | (weather_df.year == 2025))
collision_df_filtered = collision_df.filter((collision_df.year == 2023) | (collision_df.year == 2024) | (collision_df.year == 2025))

In [6]:
#extract time
collision_df_filtered = collision_df_filtered.withColumn("crash_hour", split(col("CRASH TIME"), ":")[0].cast("int"))

weather_hourly = weather_df_filtered.withColumn("weather_hour", split(col("Time"), ":")[0].cast("int"))

In [7]:
#join date+hour
joined_df = collision_df_filtered.join(weather_hourly.drop("year","month"), 
    (collision_df_filtered["CRASH DATE"] == weather_hourly["Date"]) & (collision_df_filtered["crash_hour"] == weather_hourly["weather_hour"]), "inner")

In [8]:
#extractiing hour
df_lightHour = joined_df.withColumn(
    "sunrise_hour", floor(col("daily_sunrise") / 100).cast("int")).withColumn("sunset_hour", floor(col("daily_sunset") / 100).cast("int"))

In [9]:
#get twighlight
df_lightCondition = df_lightHour.withColumn(
    "light_condition",
    when((col("crash_hour") == col("sunrise_hour")) | (col("crash_hour") == col("sunset_hour")), "Twilight")
    .when((col("crash_hour") > col("sunrise_hour")) & (col("crash_hour") < col("sunset_hour")), "Daylight")
    .otherwise("Night")
)

In [10]:
#isolating weather codess
weather_raw_df = spark.read.parquet(f"{base_path}/weather_partitioned/")
weather_raw_filtered = weather_raw_df.filter((weather_raw_df.year == 2023) | (weather_raw_df.year == 2024) | (weather_raw_df.year == 2025))

daily_weather_codes = weather_raw_filtered.filter(trim(col("REPORT_TYPE")) == "SOD") \
    .withColumn("Date", to_date(col("DATE"))) \
    .select(col("Date"), col("DailyWeather").alias("day_weather_code")) \
    .filter(col("day_weather_code").isNotNull()) \
    .filter(trim(col("day_weather_code")) != "") \
    .dropDuplicates(["Date"])

In [11]:
# --- SANITY CHECK ---
# Wir lassen uns kurz anzeigen, ob überhaupt Daten gefunden wurden!
print("=== GEFUNDENE WETTER-CODES AUS SOD ===")
daily_weather_codes.show(5)

=== GEFUNDENE WETTER-CODES AUS SOD ===
+----------+----------------+
|      Date|day_weather_code|
+----------+----------------+
|2023-01-01|           BR RA|
|2023-01-02|              RA|
|2023-01-03|     BR FG HZ RA|
|2023-01-04|           BR RA|
|2023-01-05|           BR RA|
+----------+----------------+
only showing top 5 rows



In [12]:
df_lightCondition = df_lightCondition.join(daily_weather_codes, "Date", "left")

#fix null vals
safe_weather = coalesce(col("day_weather_code"), lit(""))


#get NOAA Metar-Codes
df_lightCondition = df_lightCondition.withColumn("has_precip", safe_weather.rlike("RA|SN|DZ")) \
                         .withColumn("has_low_vis", safe_weather.rlike("FG|BR|HZ"))

df_lightCondition = df_lightCondition.withColumn(
    "hazard_category",
    when(col("has_precip") & col("has_low_vis"), "Precipitation & Low Vis")
    .when(col("has_low_vis"), "Low Visibility Only")
    .when(col("has_precip"), "Precipitation Only")
    .otherwise("Clear / Normal")
)

In [13]:
twilight_visibility_result = df_lightCondition.groupBy(
    "year", 
    "month", 
    "light_condition", 
    "hazard_category"
).agg(
    count("*").alias("total_collisions")
).orderBy("year", "month", "light_condition", "hazard_category")

twilight_visibility_result.show(100)

+----+-----+---------------+--------------------+----------------+
|year|month|light_condition|     hazard_category|total_collisions|
+----+-----+---------------+--------------------+----------------+
|2023|    1|       Daylight|      Clear / Normal|             973|
|2023|    1|       Daylight|Precipitation & L...|            2512|
|2023|    1|       Daylight|  Precipitation Only|             304|
|2023|    1|          Night|      Clear / Normal|            1318|
|2023|    1|          Night|Precipitation & L...|            3076|
|2023|    1|          Night|  Precipitation Only|             393|
|2023|    1|       Twilight|      Clear / Normal|             222|
|2023|    1|       Twilight|Precipitation & L...|             503|
|2023|    1|       Twilight|  Precipitation Only|              92|
|2023|    2|       Daylight|      Clear / Normal|            1663|
|2023|    2|       Daylight| Low Visibility Only|             400|
|2023|    2|       Daylight|Precipitation & L...|            1

In [14]:
twilight_visibility_result.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "visibility_accidents") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()